# Eval Viewer
Interactive notebook to visualize per-video predictions from a trained model.

1. Set `RESULTS_DIR` to a completed training run (the folder containing `config.yml` and fold subfolders).
2. Run all cells.
3. Use the slider to browse videos and see segment predictions + probability curves.

In [ ]:
# ── Config ──────────────────────────────────────────────
RESULTS_DIR = "/code/jjiang23/BalanceTestThesis/results/MAMP/MB/downsamp/ASFormer/20260530_215628"
DEVICE = "cuda"        # or "cpu"
STRIDE_OVERRIDE = 30   # smaller stride = smoother stitching (None = use data config stride)

In [ ]:
import os, sys, json, yaml
import importlib.util
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(RESULTS_DIR), "..", "..", "..", "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils.eval.metric_utils import predict_video, compute_segmentation_metrics, extract_segments

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ── Load config ─────────────────────────────────────────
config_path = os.path.join(RESULTS_DIR, "config.yml")
with open(config_path) as f:
    config = yaml.safe_load(f)

e_cfg = config["encoder"]
d_cfg = config["data"]
t_cfg = config["trainer"]
s_cfg = config["segmentor"]
splits_path = config["paths"]["splits_path"]

with open(splits_path) as f:
    splits = json.load(f)

print(f"Loaded config from: {config_path}")
print(f"Folds: {list(splits.keys())}")

In [ ]:
# ── Dynamic import helper ───────────────────────────────
def load_module_from_path(file_path):
    spec = importlib.util.spec_from_file_location("mod", file_path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

# ── Load initializers ───────────────────────────────────
if e_cfg is not None:
    init_encoder_path = os.path.abspath(e_cfg["init_encoder_path"])
else:
    init_encoder_path = os.path.join(PROJECT_ROOT, "initializers", "encoder", "Identity.py")

init_segmentor_path = os.path.abspath(s_cfg["init_segmentor_path"])

initialize_encoder = getattr(load_module_from_path(init_encoder_path), "initialize_encoder")
initialize_segmentor = getattr(load_module_from_path(init_segmentor_path), "initialize_segmentor")

# ── Build models (architecture only — weights loaded per fold) ──
encoder = initialize_encoder(d_cfg, e_cfg)
segmentor = initialize_segmentor(
    s_cfg, encoder,
    class_weights=None,
    lambda_smooth=t_cfg.get("lambda_smooth", 0.01),
    time_alignment=t_cfg.get("time_alignment", "downsample_labels"),
)
encoder.to(DEVICE).eval()
segmentor.to(DEVICE).eval()
print("Models ready.")

In [ ]:
# ── Collect all (fold, video) pairs ─────────────────────
video_entries = []  # list of (fold_name, video_path, fold_dir)

for fold_name, split_files in splits.items():
    fold_dir = os.path.join(RESULTS_DIR, fold_name)
    enc_ckpt = os.path.join(fold_dir, "best_encoder.pt")
    seg_ckpt = os.path.join(fold_dir, "best_segmentor.pt")
    if not (os.path.exists(enc_ckpt) and os.path.exists(seg_ckpt)):
        print(f"Skipping {fold_name}: no checkpoints")
        continue
    for vid_path in split_files["val"]:
        video_entries.append((fold_name, vid_path, fold_dir))

print(f"Total eval videos: {len(video_entries)} across {len(splits)} folds")

In [ ]:
# ── Cache for loaded fold weights ──────────────────────
_loaded_fold = None

def load_fold_weights(fold_dir):
    """Load encoder + segmentor weights for a fold (cached)."""
    global _loaded_fold
    if _loaded_fold == fold_dir:
        return
    encoder.load_state_dict(
        torch.load(os.path.join(fold_dir, "best_encoder.pt"), map_location=DEVICE)
    )
    segmentor.load_state_dict(
        torch.load(os.path.join(fold_dir, "best_segmentor.pt"), map_location=DEVICE),
        strict=False,
    )
    encoder.eval()
    segmentor.eval()
    _loaded_fold = fold_dir

In [ ]:
# ── Visualization function ─────────────────────────────
CLASS_COLORS = ["#2196F3", "#FF5722", "#4CAF50", "#FFC107", "#9C27B0"]
CLASS_NAMES = ["Phase1", "Phase2", "Phase3", "Phase4", "nonphase"]


def plot_video(idx):
    fold_name, vid_path, fold_dir = video_entries[idx]
    load_fold_weights(fold_dir)

    gt_labels, pred_labels, pred_probs = predict_video(
        vid_path, encoder, segmentor, d_cfg, DEVICE,
        stride_override=STRIDE_OVERRIDE,
    )

    T = len(gt_labels)
    frames = np.arange(T)
    num_classes = pred_probs.shape[1]
    vid_name = os.path.basename(vid_path)

    # Per-video metrics
    metrics = compute_segmentation_metrics(
        gt_labels, pred_labels, class_id=0,
        iou_thresholds=(0.1, 0.25, 0.5), fps=30,
    )
    valid = gt_labels != -100
    frame_acc = (gt_labels[valid] == pred_labels[valid]).mean() if valid.sum() > 0 else 0.0

    # ── Figure ──
    fig, axes = plt.subplots(
        3, 1, figsize=(18, 8), sharex=True,
        gridspec_kw={"height_ratios": [1, 1, 2.5], "hspace": 0.08},
    )

    # --- Row 1: Ground truth color bar ---
    ax_gt = axes[0]
    for c in range(num_classes):
        mask_c = (gt_labels == c)
        ax_gt.fill_between(frames, 0, 1, where=mask_c,
                           color=CLASS_COLORS[c % len(CLASS_COLORS)], alpha=0.9,
                           label=CLASS_NAMES[c] if c < len(CLASS_NAMES) else f"Class {c}")
    ax_gt.set_yticks([])
    ax_gt.set_ylabel("GT", fontsize=10, fontweight="bold")
    ax_gt.legend(loc="upper right", fontsize=7, ncol=num_classes)

    # --- Row 2: Prediction color bar ---
    ax_pred = axes[1]
    for c in range(num_classes):
        mask_c = (pred_labels == c)
        ax_pred.fill_between(frames, 0, 1, where=mask_c,
                             color=CLASS_COLORS[c % len(CLASS_COLORS)], alpha=0.9)
    ax_pred.set_yticks([])
    ax_pred.set_ylabel("Pred", fontsize=10, fontweight="bold")

    # --- Row 3: Probability curves ---
    ax_prob = axes[2]
    for c in range(num_classes):
        label = CLASS_NAMES[c] if c < len(CLASS_NAMES) else f"Class {c}"
        ax_prob.plot(frames, pred_probs[:, c], label=label,
                     color=CLASS_COLORS[c % len(CLASS_COLORS)], lw=1.2)
    ax_prob.set_ylim(-0.05, 1.05)
    ax_prob.set_ylabel("Probability", fontsize=10)
    ax_prob.set_xlabel("Frame", fontsize=10)
    ax_prob.legend(loc="upper right", fontsize=7)
    ax_prob.grid(axis="y", alpha=0.3)

    # --- Title with metrics ---
    f1_10 = metrics.get("f1_iou_0.1", 0)
    f1_25 = metrics.get("f1_iou_0.25", 0)
    f1_50 = metrics.get("f1_iou_0.5", 0)
    fig.suptitle(
        f"[{fold_name}] {vid_name}   |   "
        f"Acc: {frame_acc:.1%}   "
        f"F1@.10: {f1_10:.2f}  F1@.25: {f1_25:.2f}  F1@.50: {f1_50:.2f}   "
        f"({T} frames)",
        fontsize=11, fontweight="bold", y=1.01,
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ── Interactive slider ─────────────────────────────────
from ipywidgets import interact, IntSlider

slider = IntSlider(
    value=0, min=0, max=len(video_entries) - 1, step=1,
    description="Video:",
    continuous_update=False,
    style={"description_width": "60px"},
    layout={"width": "600px"},
)

interact(plot_video, idx=slider);